<a href="https://colab.research.google.com/github/Bhagya-Navodyani/Intelligent_Fish_Storage-ML/blob/main/notebook/Fish_Storage_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Dataset Loading

# Intelligent Fish Storage ML Pipeline

**Research Section:** Intelligent Fish Storage Quality Monitoring and Remaining Shelf-Life Prediction Using IoT and Machine Learning

This notebook contains the preprocessing and machine learning workflow for sensor data collected using the ESP32-S3 fish-storage monitoring prototype.

In [17]:
import pandas as pd
import numpy as np

# CSV file already uploaded to Colab
filename = "/content/Final_csv.csv"

# Read CSV
df_raw = pd.read_csv(filename)

print("Uploaded file:", filename)
print("Rows, Columns:", df_raw.shape)

print("\nColumns:")
print(df_raw.columns.tolist())

df_raw.head()

Uploaded file: /content/Final_csv.csv
Rows, Columns: (10465, 20)

Columns:
['pc_timestamp', 'trial_id', 'fish_species', 'record_type', 'elapsed_sec', 'storage_temp_c', 'air_temp_c', 'humidity_pct', 'distance_cm', 'gas_raw', 'reed_raw', 'lid', 'current_lid_open_sec', 'total_lid_open_sec', 'lid_open_count', 'storage_time_hours', 'temp_difference_c', 'lid_open_ratio', 'lid_open_rate_per_hour', 'avg_lid_open_duration_sec']


,pc_timestamp,trial_id,fish_species,record_type,elapsed_sec,storage_temp_c,air_temp_c,humidity_pct,distance_cm,gas_raw,reed_raw,lid,current_lid_open_sec,total_lid_open_sec,lid_open_count,storage_time_hours,temp_difference_c,lid_open_ratio,lid_open_rate_per_hour,avg_lid_open_duration_sec
0,2026-09-29 10:02:23,TRIAL_20260929_100223,Linna,START,0,0.25,28.5,84.8,28.37,835,0,OPEN,0,0,1,0.0000,28.25,0.0,0.0,0.0
1,2026-09-29 10:02:53,TRIAL_20260929_100223,Linna,PERIODIC,30,0.25,28.3,84.1,28.38,835,0,OPEN,30,30,1,0.0083,28.05,1.0,0.0,30.0
2,2026-09-29 10:03:23,TRIAL_20260929_100223,Linna,PERIODIC,60,0.25,28.2,86.0,28.38,835,0,OPEN,60,60,1,0.0167,27.95,1.0,0.0,60.0
3,2026-09-29 10:03:53,TRIAL_20260929_100223,Linna,PERIODIC,90,0.25,28.1,86.0,28.37,835,0,OPEN,90,90,1,0.0250,27.85,1.0,0.0,90.0
4,2026-09-29 10:04:23,TRIAL_20260929_100223,Linna,PERIODIC,120,0.25,28.0,86.4,29.26,835,0,OPEN,120,120,1,0.0333,27.75,1.0,0.0,120.0


## 2. Dataset Audit

In [18]:
# Create a working copy while preserving the loaded dataset
df = df_raw.copy()

# Convert timestamp to datetime format
df["pc_timestamp"] = pd.to_datetime(
    df["pc_timestamp"],
    errors="coerce"
)

# Sort records chronologically
df = df.sort_values("pc_timestamp").reset_index(drop=True)

# Dataset summary
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nFirst timestamp:", df["pc_timestamp"].min())
print("Last timestamp :", df["pc_timestamp"].max())

display(df.head())

Total rows: 10465
Total columns: 20

Missing values in each column:
pc_timestamp                   0
trial_id                       0
fish_species                   0
record_type                    0
elapsed_sec                    0
storage_temp_c                 0
air_temp_c                     0
humidity_pct                   0
distance_cm                  311
gas_raw                        0
reed_raw                       0
lid                            0
current_lid_open_sec           0
total_lid_open_sec             0
lid_open_count                 0
storage_time_hours             0
temp_difference_c              0
lid_open_ratio                 0
lid_open_rate_per_hour         0
avg_lid_open_duration_sec      0
dtype: int64

First timestamp: 2026-09-29 10:02:23
Last timestamp : 2026-10-03 00:22:24


,pc_timestamp,trial_id,fish_species,record_type,elapsed_sec,storage_temp_c,air_temp_c,humidity_pct,distance_cm,gas_raw,reed_raw,lid,current_lid_open_sec,total_lid_open_sec,lid_open_count,storage_time_hours,temp_difference_c,lid_open_ratio,lid_open_rate_per_hour,avg_lid_open_duration_sec
0,2026-09-29 10:02:23,TRIAL_20260929_100223,Linna,START,0,0.25,28.5,84.8,28.37,835,0,OPEN,0,0,1,0.0000,28.25,0.0,0.0,0.0
1,2026-09-29 10:02:53,TRIAL_20260929_100223,Linna,PERIODIC,30,0.25,28.3,84.1,28.38,835,0,OPEN,30,30,1,0.0083,28.05,1.0,0.0,30.0
2,2026-09-29 10:03:23,TRIAL_20260929_100223,Linna,PERIODIC,60,0.25,28.2,86.0,28.38,835,0,OPEN,60,60,1,0.0167,27.95,1.0,0.0,60.0
3,2026-09-29 10:03:53,TRIAL_20260929_100223,Linna,PERIODIC,90,0.25,28.1,86.0,28.37,835,0,OPEN,90,90,1,0.0250,27.85,1.0,0.0,90.0
4,2026-09-29 10:04:23,TRIAL_20260929_100223,Linna,PERIODIC,120,0.25,28.0,86.4,29.26,835,0,OPEN,120,120,1,0.0333,27.75,1.0,0.0,120.0


In [19]:
# Check exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

# Display basic numerical statistics
display(df.describe())

Exact duplicate rows: 0


,pc_timestamp,elapsed_sec,storage_temp_c,air_temp_c,humidity_pct,distance_cm,gas_raw,reed_raw,current_lid_open_sec,total_lid_open_sec,lid_open_count,storage_time_hours,temp_difference_c,lid_open_ratio,lid_open_rate_per_hour,avg_lid_open_duration_sec
count,10465,10465.000000,10465.000000,10465.000000,10465.000000,10154.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000,10465.000000
mean,2026-10-01 05:07:33.561681664,155110.561682,1.828822,17.700554,81.426393,29.699733,473.731295,0.944386,19.286670,9724.143717,35.257047,43.086266,15.871732,0.080273,1.124134,265.053454
min,2026-09-29 10:02:23,0.000000,-0.250000,13.000000,57.400000,4.720000,200.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.900000,0.000000,0.000000,0.000000
25%,2026-09-30 07:29:29,77226.000000,0.000000,15.300000,73.300000,27.870000,275.000000,1.000000,0.000000,6551.000000,26.000000,21.451700,13.800000,0.054727,0.679600,266.136000
50%,2026-10-01 05:06:26,155043.000000,0.500000,17.300000,80.000000,28.710000,393.000000,1.000000,0.000000,10981.000000,37.000000,43.067500,15.650000,0.070716,0.837200,275.807000
75%,2026-10-02 02:48:26,233163.000000,1.500000,19.300000,89.200000,30.410000,627.000000,1.000000,0.000000,12762.000000,43.000000,64.767500,17.600000,0.080717,1.127900,296.784000
max,2026-10-03 00:22:24,310801.000000,23.500000,28.500000,100.000000,63.830000,1761.000000,1.000000,2278.000000,15721.000000,57.000000,86.333600,28.250000,1.000000,6.818200,630.000000
std,NaN,89930.812406,3.498060,3.010103,9.844427,3.760690,234.341593,0.229186,129.840186,4112.439112,12.845562,24.980780,3.476105,0.065577,0.872750,53.509667


### Initial Audit Findings

- The dataset contains 10,465 sensor observations.
- No exact duplicate rows were detected.
- The records are arranged chronologically using `pc_timestamp`.
- The `distance_cm` feature contains 311 missing values.
- The storage-temperature and gas-sensor columns require further anomaly investigation before model training.
- Sensor-quality issues will be handled during the preprocessing stage without modifying the original dataset.

In [20]:
# Check important sensor anomalies

zero_temp_count = (df["storage_temp_c"] == 0).sum()
high_gas_count = (df["gas_raw"] > 1000).sum()
missing_distance_count = df["distance_cm"].isnull().sum()

print("storage_temp_c = 0 rows:", zero_temp_count)
print("gas_raw > 1000 rows:", high_gas_count)
print("Missing distance_cm rows:", missing_distance_count)

storage_temp_c = 0 rows: 3128
gas_raw > 1000 rows: 14
Missing distance_cm rows: 311


## 3. Data Preprocessing

This section prepares the audited sensor dataset for machine learning while preserving the original measurements and documenting each preprocessing step.

### 3.1 Handling Missing Distance Measurements

Missing `distance_cm` values are handled in a separate cleaned feature while preserving the original measurements.

In [21]:
# Preserve the original distance measurements
df["distance_cm_clean"] = df["distance_cm"].copy()

# Interpolate missing values using neighboring time-series observations
df["distance_cm_clean"] = (
    df["distance_cm_clean"]
    .interpolate(method="linear")
    .bfill()
    .ffill()
)

print("Original missing distance values:",
      df["distance_cm"].isnull().sum())

print("Missing values after preprocessing:",
      df["distance_cm_clean"].isnull().sum())

Original missing distance values: 311
Missing values after preprocessing: 0


### 3.2 Storage Temperature Anomaly Investigation

The dataset contains a large number of zero values in `storage_temp_c`.
These readings are investigated before preprocessing to determine whether they represent valid near-ice temperatures or a sensor/wiring issue.

In [22]:
# Identify zero storage-temperature readings

zero_temp = df[df["storage_temp_c"] == 0].copy()

print("Total zero-temperature rows:", len(zero_temp))

print("\nFirst zero-temperature readings:")
display(
    zero_temp[
        ["pc_timestamp", "elapsed_sec", "storage_temp_c", "air_temp_c"]
    ].head(10)
)

print("\nLast zero-temperature readings:")
display(
    zero_temp[
        ["pc_timestamp", "elapsed_sec", "storage_temp_c", "air_temp_c"]
    ].tail(10)
)

Total zero-temperature rows: 3128

First zero-temperature readings:


,pc_timestamp,elapsed_sec,storage_temp_c,air_temp_c
12,2026-09-29 10:08:23,360,0.0,27.5
17,2026-09-29 10:10:53,510,0.0,26.3
18,2026-09-29 10:11:23,540,0.0,26.1
19,2026-09-29 10:11:53,570,0.0,25.9
20,2026-09-29 10:12:23,600,0.0,25.8
21,2026-09-29 10:12:53,630,0.0,25.6
22,2026-09-29 10:12:55,632,0.0,25.6
23,2026-09-29 10:13:23,660,0.0,25.2
24,2026-09-29 10:13:53,690,0.0,24.7
25,2026-09-29 10:14:23,720,0.0,24.2



Last zero-temperature readings:


,pc_timestamp,elapsed_sec,storage_temp_c,air_temp_c
9506,2026-10-02 16:26:24,282241,0.0,19.9
9508,2026-10-02 16:27:24,282301,0.0,19.9
9510,2026-10-02 16:28:24,282361,0.0,19.9
9511,2026-10-02 16:28:54,282391,0.0,19.9
9517,2026-10-02 16:31:54,282571,0.0,20.0
9518,2026-10-02 16:32:24,282601,0.0,20.0
9521,2026-10-02 16:33:54,282691,0.0,20.1
9528,2026-10-02 16:37:24,282901,0.0,20.1
9533,2026-10-02 16:39:54,283051,0.0,20.2
9536,2026-10-02 16:41:24,283141,0.0,20.2


### Storage Temperature Investigation Findings

- A total of 3,128 zero storage-temperature readings were detected.
- The zero readings occur across multiple separate time segments rather than one single continuous period.
- Both isolated zero readings and longer zero-reading periods are present.
- Therefore, zero values are not automatically replaced during this stage.
- Additional context and neighboring measurements will be examined before defining a valid preprocessing rule.

In [23]:
# Show the longest zero-temperature segments

longest_zero_segments = zero_segments.sort_values(
    "duration_minutes",
    ascending=False
).head(15)

display(longest_zero_segments)

,temp_zero_group,start_time,end_time,start_elapsed,end_elapsed,rows,duration_minutes
160,322,2026-10-01 20:33:26,2026-10-02 00:20:56,210663,224313,459,227.500000
146,294,2026-10-01 13:39:57,2026-10-01 16:33:27,185854,196264,351,173.500000
120,242,2026-09-30 18:25:27,2026-09-30 20:44:27,116584,124924,282,139.000000
90,182,2026-09-29 16:34:40,2026-09-29 18:51:39,23537,31756,277,136.983333
190,382,2026-10-02 04:41:55,2026-10-02 06:05:55,239972,245012,169,84.000000
139,280,2026-10-01 11:07:25,2026-10-01 12:09:27,176702,180424,125,62.033333
147,296,2026-10-01 16:34:27,2026-10-01 17:35:27,196324,199984,123,61.000000
233,468,2026-10-02 15:24:54,2026-10-02 16:05:24,278551,280981,82,40.500000
4,10,2026-09-29 10:34:23,2026-09-29 11:05:53,1920,3810,67,31.500000
223,448,2026-10-02 13:33:24,2026-10-02 14:03:24,271861,273661,61,30.000000


### Storage Temperature Anomaly Decision

Several zero-temperature periods persist continuously for long durations, including segments lasting more than two hours. These long flat-line sequences are treated as suspected sensor/wiring anomalies rather than automatically assuming that all zero readings represent true storage temperature.

The original `storage_temp_c` values are preserved. A separate validity flag will be used to distinguish suspected invalid readings from usable measurements. Long missing/faulty periods will not be replaced by arbitrary temperature values.

In [24]:
# Re-create zero-temperature groups correctly

# 1. Mark rows where storage temperature is exactly zero
df["temp_is_zero"] = (df["storage_temp_c"] == 0).astype(int)

# 2. Create continuous segment IDs
df["temp_zero_group"] = (
    df["temp_is_zero"]
    .ne(df["temp_is_zero"].shift())
    .cumsum()
)

# 3. Create zero-segment summary
# IMPORTANT: reset_index() keeps temp_zero_group as a column
zero_segments = (
    df[df["temp_is_zero"] == 1]
    .groupby("temp_zero_group")
    .agg(
        start_time=("pc_timestamp", "first"),
        end_time=("pc_timestamp", "last"),
        start_elapsed=("elapsed_sec", "first"),
        end_elapsed=("elapsed_sec", "last"),
        rows=("storage_temp_c", "size")
    )
    .reset_index()
)

# 4. Calculate segment duration
zero_segments["duration_minutes"] = (
    zero_segments["end_elapsed"] -
    zero_segments["start_elapsed"]
) / 60

display(zero_segments.head())

,temp_zero_group,start_time,end_time,start_elapsed,end_elapsed,rows,duration_minutes
0,2,2026-09-29 10:08:23,2026-09-29 10:08:23,360,360,1,0.000000
1,4,2026-09-29 10:10:53,2026-09-29 10:23:23,510,1260,28,12.500000
2,6,2026-09-29 10:27:23,2026-09-29 10:27:53,1500,1530,2,0.500000
3,8,2026-09-29 10:29:23,2026-09-29 10:33:06,1620,1843,10,3.716667
4,10,2026-09-29 10:34:23,2026-09-29 11:05:53,1920,3810,67,31.500000


In [25]:
# Preserve the original storage temperature
df["storage_temp_original_c"] = df["storage_temp_c"].copy()

# Create a quality flag
df["storage_temp_valid"] = 1

# Identify long zero flat-line segments (>= 5 minutes)
long_zero_groups = zero_segments.loc[
    zero_segments["duration_minutes"] >= 5,
    "temp_zero_group"
]

# Mark these periods as suspected invalid
df.loc[
    df["temp_zero_group"].isin(long_zero_groups),
    "storage_temp_valid"
] = 0

print(
    "Valid temperature rows:",
    (df["storage_temp_valid"] == 1).sum()
)

print(
    "Suspected invalid temperature rows:",
    (df["storage_temp_valid"] == 0).sum()
)

Valid temperature rows: 7822
Suspected invalid temperature rows: 2643
